# UrbanFlow Green Mobility Intelligence ML Project

###### *Epsilon AI | Course Final Project | Developed by **`Ayman Nader`***

## Project Overview
UrbanFlow Green Mobility Intelligence is an enterprise-grade MLOps data product and decision-support web application built with Streamlit. Designed for modern smart-city traffic management and urban planning, the platform delivers multi-target predictive insights—including real-time congestion classification, intersection wait time regression, and carbon emission modeling—empowered by robust XGBoost pipelines, automated artifact deserialization, local explainability metrics, and dynamic executive report generation.

---

## Dataset Source & Documentation
* **Source Datacard:** [CityFlow - Smart Urban Mobility & Traffic IoT on Kaggle](https://www.kaggle.com/datasets/mobeenfatimah/cityflow-smart-urban-mobility-and-traffic-iot)
* **Description:** The dataset models high-dimensional urban traffic parameters, IoT sensor network metrics, environmental indicators, and temporal flags across metropolitan road segments. It serves as an empirical foundation for multi-target regression and classification tasks aimed at reducing urban congestion and optimizing green transit infrastructure.

---

## Citation & License
* **Dataset Attribution:** Mobeen Fatimah, *CityFlow - Smart Urban Mobility & Traffic IoT Dataset*, Kaggle.
* **Data License:** CC BY 4.0 (Open Access) and subject to Kaggle's standard platform data use terms and the specific repository sharing permissions provided by the author. Intended for research, ML benchmark tasks, and smart-city dashboards.

## Packages & Libraries Setup

In [2]:

# ==========================================
# CORE DATA MANIPULATION & NUMERICS
# ==========================================
import numpy as np
import pandas as pd

# ==========================================
# INTERACTIVE VISUALIZATION (Plotly)
# ==========================================
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Set default Plotly theme
pio.templates.default = "plotly_white"

# ==========================================
# MACHINE LEARNING: PREPROCESSING & METRICS
# ==========================================
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.preprocessing import PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Evaluation Metrics
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    roc_auc_score, mean_squared_error, mean_absolute_error, r2_score, f1_score
)

# =============================
# MACHINE LEARNING: ALGORITHMS
# =============================

# Classification Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Regression Models
from sklearn.linear_model import LinearRegression, SGDRegressor, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# Saving & Retaining models
import joblib

# Create requirements.txt
import importlib.metadata
import sys

print("All required libraries and tools imported successfully!")


All required libraries and tools imported successfully!


* We first turned the .csv dataset file into .parquet by using `to_parquet` command.
 
 **Why?** >> CSVs are excellent for quick, human-readable inspections of tiny datasets, Parquet excels in speed, storage efficiency, and data integrity. It saves disk space and drastically reduce cloud storage costs, and is best when building a repeatable Machine Learning pipeline where data type consistency is critical.

In [2]:
# df = pd.read_csv('smart_city_traffic_mobility.csv')
# df.to_parquet('smart_city_traffic_mobility.parquet')

In [5]:
df = pd.read_parquet('smart_city_traffic_mobility.parquet')

# Display total rows and columns
print(f"Dataset Successfully Loaded!")
print(f"Total Rows (Incident Records): {df.shape[0]:,}")
print(f"Total Features (Columns): {df.shape[1]}")

# Preview the first 3 records
df.head(3)

Dataset Successfully Loaded!
Total Rows (Incident Records): 204,000
Total Features (Columns): 47


,record_id,city_zone,road_id,intersection_id,latitude,longitude,road_type,lanes,speed_limit,vehicle_count,...,traffic_camera_count,iot_sensor_health,signal_cycle_seconds,green_light_duration,emergency_vehicle_detected,traffic_flow_rate,congestion_score,emission_estimate,fuel_waste_estimate,congestion_level
0,REC_0000001,Financial District,RD_206,INT_001,40.757551,-73.990504,Local Street,2,40,162,...,1,Optimal,60,18,0,162.0,16.8,39.14,8.53,Low
1,REC_0000002,Financial District,RD_206,INT_001,40.757551,-73.990504,Local Street,2,40,150,...,1,Optimal,60,18,0,150.0,18.4,36.09,7.83,Low
2,REC_0000003,Financial District,RD_206,INT_001,40.757551,-73.990504,Local Street,2,40,186,...,1,Optimal,60,18,0,186.0,20.7,44.01,9.39,Low


### Data Dictionary

| Category | Column Name | Data Type | Description |
| :--- | :--- | :--- | :--- |
| **Spatial & Infrastructure** | `record_id` | String | Unique observation identifier (`REC_XXXXXXX`) |
| **Spatial & Infrastructure** | `city_zone` | Categorical | Geographic and functional district within the city (e.g., Financial District, Industrial East, Suburban North) |
| **Spatial & Infrastructure** | `road_id` | String | Anonymized unique identifier for the specific road segment (`RD_XXX`) |
| **Spatial & Infrastructure** | `intersection_id` | String | Fixed smart-city traffic junction node identifier (`INT_XXX`) |
| **Spatial & Infrastructure** | `latitude` | Float | Exact spatial WGS84 geographic latitude coordinate of the intersection |
| **Spatial & Infrastructure** | `longitude` | Float | Exact spatial WGS84 geographic longitude coordinate of the intersection |
| **Spatial & Infrastructure** | `road_type` | Categorical | Classification of roadway governing baseline capacity (e.g., Local Street, Arterial, Collector, Highway) |
| **Spatial & Infrastructure** | `lanes` | Integer | Total active driving lanes available across both directions |
| **Spatial & Infrastructure** | `speed_limit` | Integer | Maximum legally permitted speed on the roadway segment in km/h |
| **Traffic Flow Dynamics** | `vehicle_count` | Integer | Total number of all motorized vehicles detected passing through the intersection |
| **Traffic Flow Dynamics** | `average_speed` | Float | Mean observed traffic speed in km/h |
| **Traffic Flow Dynamics** | `heavy_vehicle_count` | Integer | Total count of detected trucks and buses |
| **Traffic Flow Dynamics** | `motorcycle_count` | Integer | Total count of detected two-wheelers / motorcycles |
| **Traffic Flow Dynamics** | `public_transport_count` | Integer | Total count of detected public transit city buses |
| **Traffic Flow Dynamics** | `traffic_density` | Float | Traffic density measured in vehicles per kilometer per lane |
| **Traffic Flow Dynamics** | `queue_length` | Float | Estimated tailback queue length in meters |
| **Traffic Flow Dynamics** | `average_wait_time` | Float | Average delay per vehicle at the intersection in seconds |
| **Traffic Flow Dynamics** | `traffic_flow_rate` | Float | Rate of traffic flow passing through the junction |
| **Temporal Features** | `timestamp` | Datetime | ISO hourly observation timestamp (`YYYY-MM-DD HH:MM:SS`) |
| **Temporal Features** | `hour` | Integer | Hour of the day (`0` to `23`) |
| **Temporal Features** | `day_of_week` | Integer | Day index (`0` = Monday through `6` = Sunday) |
| **Temporal Features** | `is_weekend` | Binary | Flag indicating weekend (`1` if Saturday/Sunday; `0` otherwise) |
| **Temporal Features** | `is_holiday` | Binary | Flag indicating official public holiday (`1` if holiday; `0` otherwise) |
| **Temporal Features** | `rush_hour` | Binary | Flag indicating peak commuting hours (`1` during rush hour; `0` otherwise) |
| **Temporal Features** | `peak_period` | Categorical | Period classification bin (e.g., Night, Off-Peak, Morning Peak, Evening Peak) |
| **Environmental & Weather** | `temperature` | Float | Ambient air temperature in degrees Celsius (°C) |
| **Environmental & Weather** | `rainfall_mm` | Float | Hourly precipitation depth in millimeters (mm) |
| **Environmental & Weather** | `visibility_km` | Float | Meteorological visibility distance in kilometers (km) |
| **Environmental & Weather** | `air_quality_index` | Integer | Recorded Air Quality Index (AQI) value |
| **Environmental & Weather** | `weather_condition` | Categorical | General weather state (e.g., Clear, Rain, Heavy Rain, Fog) |
| **Urban Context & Infrastructure** | `nearby_school` | Binary | Indicator for educational institution proximity (`1` = yes; `0` = no) |
| **Urban Context & Infrastructure** | `nearby_hospital` | Binary | Indicator for medical facility proximity (`1` = yes; `0` = no) |
| **Urban Context & Infrastructure** | `nearby_market` | Binary | Indicator for commercial market proximity (`1` = yes; `0` = no) |
| **Urban Context & Infrastructure** | `construction_activity` | Binary | Flag indicating active road construction nearby (`1` = yes; `0` = no) |
| **Urban Context & Infrastructure** | `road_condition` | Categorical | Physical surface state of the roadway (e.g., Good, Poor, Wet) |
| **Urban Context & Infrastructure** | `parking_occupancy` | Float | Percentage or occupancy metric of nearby parking facilities |
| **Urban Context & Infrastructure** | `public_event` | Binary | Flag indicating public event taking place nearby (`1` = yes; `0` = no) |
| **Urban Context & Infrastructure** | `accident_reported` | Binary | Flag indicating traffic accident reported at or near the intersection (`1` = yes; `0` = no) |
| **IoT Sensors & Operations** | `traffic_camera_count` | Integer | Total number of operational smart traffic cameras monitoring the junction |
| **IoT Sensors & Operations** | `iot_sensor_health` | Categorical | Operational status of IoT sensors (e.g., Optimal, Degraded, Faulty) |
| **IoT Sensors & Operations** | `signal_cycle_seconds` | Integer | Total duration of a complete traffic light signal cycle in seconds |
| **IoT Sensors & Operations** | `green_light_duration` | Integer | Duration of the green light signal phase in seconds |
| **IoT Sensors & Operations** | `emergency_vehicle_detected` | Binary | Flag indicating emergency vehicle preemption/detection (`1` = yes; `0` = no) |
| **Traffic Impact & Derived Metrics** | `congestion_score` | Float | Computed composite score representing traffic congestion severity |
| **Traffic Impact & Derived Metrics** | `emission_estimate` | Float | Estimated vehicular carbon/pollutant emissions (in kg) |
| **Traffic Impact & Derived Metrics** | `fuel_waste_estimate` | Float | Estimated fuel wastage due to idling and congestion (in liters) |
| **Traffic Impact & Derived Metrics** | `congestion_level` | Categorical | Categorical label rating congestion severity (e.g., Low, High, Severe) |

## Data Inspection

In [ ]:
# ===============================
# DATA INSPECTION & HEALTH CHECK
# ===============================

print(f"Dataset Dimensions: {df.shape[0]:,} rows | {df.shape[1]:,} columns\n")

print("=== 1. DATA TYPES & MEMORY USAGE ===")
print(df.dtypes.value_counts())
print(f"Total Memory Footprint: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB\n")

print("=== 2. MISSING VALUES & COMPLETENESS CHECK ===")
missing_df = pd.DataFrame({
    'Missing Count': df.isna().sum(),
    'Missing Percentage (%)': (df.isna().mean() * 100).round(2)
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values(by='Missing Count', ascending=False)

if not missing_df.empty:
    print(missing_df.to_string())
    total_cells = df.size
    total_missing = df.isnull().sum().sum()
    total_percentage = (total_missing / total_cells) * 100
    print(f"\nTotal percentage of missing data across dataset: {total_percentage:.2f}%")
else:
    print("No missing values detected across the dataset.")

print("\n=== 3. DUPLICATE RECORDS CHECK ===")
duplicate_rows = df.duplicated().sum()
print(f"Total Duplicate Rows: {duplicate_rows:,}\n")

print("=== 4. NUMERICAL FEATURES SUMMARY & SKEWNESS ===")
num_cols = df.select_dtypes(include=[np.number]).columns
if len(num_cols) > 0:
    num_summary = df[num_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]
    num_summary['skewness'] = df[num_cols].skew()
    print(num_summary.to_string())
else:
    print("No numerical columns found.")

print("\n=== 5. CATEGORICAL FEATURES CARDINALITY ===")
cat_cols = df.select_dtypes(include=['str', 'category']).columns
if len(cat_cols) > 0:
    cat_summary = pd.DataFrame({
        'Unique Categories': df[cat_cols].nunique(),
        'Most Frequent Value': [df[col].mode().iloc[0] if not df[col].mode().empty else np.nan for col in cat_cols],
        'Dominant Class %': [(df[col].value_counts(normalize=True).iloc[0] * 100).round(2) if not df[col].empty else 0 for col in cat_cols]
    }, index=cat_cols)
    print(cat_summary.to_string())
else:
    print("No categorical columns found.")

print("\n=== 6. TARGET VARIABLES HEALTH & DISTRIBUTION ===")
# Check Classification Target Candidates
for target_cand in ['congestion_level']:
    if target_cand in df.columns:
        print(f"\nClassification Target [{target_cand}] Breakdown:")
        counts = df[target_cand].value_counts(dropna=False)
        pcts = df[target_cand].value_counts(normalize=True, dropna=False).mul(100).round(2)
        target_dist = pd.DataFrame({'Count': counts, 'Percentage (%)': pcts})
        print(target_dist.to_string())

# Check Regression Target Candidates (e.g., Wait Time, Carbon Emissions)
for reg_cand in ['average_wait_time','emission_estimate','queue_length']:
    if reg_cand in df.columns:
        print(f"\nRegression Target [{reg_cand}] Summary:")
        print(df[reg_cand].describe())

print("\n=== 7. PREVIEW OF FIRST 3 ROWS ===")
display(df.head(3))

Dataset Dimensions: 204,000 rows | 47 columns

=== 1. DATA TYPES & MEMORY USAGE ===
int64      22
float64    14
str        11
Name: count, dtype: int64
Total Memory Footprint: 91.56 MB

=== 2. MISSING VALUES & COMPLETENESS CHECK ===
No missing values detected across the dataset.

=== 3. DUPLICATE RECORDS CHECK ===
Total Duplicate Rows: 0

=== 4. NUMERICAL FEATURES SUMMARY & SKEWNESS ===
                                  mean         std        min         50%           max  skewness
latitude                     40.716219    0.044876  40.636691   40.721824     40.790642 -0.094169
longitude                   -74.004859    0.044211 -74.083673  -74.001825    -73.928036 -0.042733
lanes                         2.750000    0.726294   2.000000    3.000000      4.000000  0.420891
speed_limit                  56.700000   17.609984  40.000000   60.000000     90.000000  0.794002
vehicle_count               749.585907  754.542368  54.000000  308.000000   3162.000000  1.124276
average_speed         

,record_id,city_zone,road_id,intersection_id,latitude,longitude,road_type,lanes,speed_limit,vehicle_count,...,traffic_camera_count,iot_sensor_health,signal_cycle_seconds,green_light_duration,emergency_vehicle_detected,traffic_flow_rate,congestion_score,emission_estimate,fuel_waste_estimate,congestion_level
0,REC_0000001,Financial District,RD_206,INT_001,40.757551,-73.990504,Local Street,2,40,162,...,1,Optimal,60,18,0,162.0,16.8,39.14,8.53,Low
1,REC_0000002,Financial District,RD_206,INT_001,40.757551,-73.990504,Local Street,2,40,150,...,1,Optimal,60,18,0,150.0,18.4,36.09,7.83,Low
2,REC_0000003,Financial District,RD_206,INT_001,40.757551,-73.990504,Local Street,2,40,186,...,1,Optimal,60,18,0,186.0,20.7,44.01,9.39,Low


### General Findings & Observations
1. **Feature Cardinality & Identifiers:** `record_id` contains 204,000 unique keys and must be dropped to prevent model overfitting. Spatial IDs like `road_id` (94) and `intersection_id` (100) will require structured encoding.
2. **Environmental & Sensor Skew:** 
   - `weather_condition` is heavily dominated by **Clear** (90.03%).
   - `road_condition` is mostly **Good** (86.98%).
   - `iot_sensor_health` is **Optimal** in 93.0% of records. 
   *(Insight: The dataset predominantly captures baseline normal traffic conditions with specific operational spike anomalies).*
3. **Target Distribution Health:**
   - **Classification Target (`congestion_level`):** Well-distributed across 4 tiers (**Low**: 49.84%, **Severe**: 35.05%, **Moderate**: 7.83%, **High**: 7.28%). The strong representation of Severe congestion ensures robust classifier training.
   - **Regression Targets (`average_wait_time`, `emission_estimate`, `queue_length`):** All three targets exhibit **extreme right-skewness** (e.g., `average_wait_time` has a median of 1.9 minutes but a max of 1,174.5 minutes). Log transformations (`np.log1p`) will be necessary for stable regressor convergence.

---

In [6]:
# ==========================================
# IN-DEPTH DATA INSPECTION FOR MULTI-TARGET MODELING
# ==========================================

print(f"Dataset Shape: {df.shape[0]:,} rows | {df.shape[1]:,} columns\n")

print("=" * 60)
print("1. CATEGORICAL COLUMNS IN-DEPTH CHECK")
print("=" * 60)

cat_cols_preview = df.select_dtypes(include=['str', 'category']).columns.tolist()
print(cat_cols_preview)

cat_cols = df.select_dtypes(include=['str', 'category']).columns

for col in cat_cols:
    print(f"\nFeature: '{col}'")
    print(f"• Unique Values: {df[col].nunique():,}")
    print(f"• Missing Values: {df[col].isna().sum():,} ({(df[col].isna().mean() * 100):.2f}%)")
    print("• Top Value Distribution:")
    print(df[col].value_counts(dropna=False).head(5).to_string())
    print("-" * 60)

print("\n" + "=" * 60)
print("2. NUMERICAL COLUMNS IN-DEPTH CHECK & SKEWNESS")
print("=" * 60)

num_cols_preview = df.select_dtypes(include=[np.number]).columns.tolist()
print(num_cols_preview)

num_cols = df.select_dtypes(include=[np.number]).columns

if len(num_cols) > 0:
    num_summary = df[num_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]
    num_summary['skewness'] = df[num_cols].skew()
    num_summary['missing_count'] = df[num_cols].isna().sum()
    num_summary['missing_pct'] = (df[num_cols].isna().mean() * 100).round(2)
    print(num_summary.to_string())
else:
    print("No numerical columns found.")

print("\n" + "=" * 60)
print("3. TARGET VARIABLES HEALTH CHECK")
print("=" * 60)

# Classification Target Candidates
class_target = 'congestion_level'
if class_target in df.columns:
    print(f"\n[Classification Target] '{class_target}':")
    print(df[class_target].value_counts(dropna=False))
    print(f"Missing Values: {df[class_target].isna().sum()}")

# Regression Target Candidates
reg_targets = ['average_wait_time', 'emission_estimate', 'queue_length']
for rt in reg_targets:
    if rt in df.columns:
        print(f"\n[Regression Target] '{rt}':")
        print(df[rt].describe())
        print(f"Missing Values: {df[rt].isna().sum()}")

Dataset Shape: 204,000 rows | 47 columns

1. CATEGORICAL COLUMNS IN-DEPTH CHECK
['record_id', 'city_zone', 'road_id', 'intersection_id', 'road_type', 'timestamp', 'peak_period', 'weather_condition', 'road_condition', 'iot_sensor_health', 'congestion_level']

Feature: 'record_id'
• Unique Values: 204,000
• Missing Values: 0 (0.00%)
• Top Value Distribution:
record_id
REC_0000001    1
REC_0000002    1
REC_0000003    1
REC_0000004    1
REC_0000005    1
------------------------------------------------------------

Feature: 'city_zone'
• Unique Values: 6
• Missing Values: 0 (0.00%)
• Top Value Distribution:
city_zone
Suburban North        53040
Financial District    48960
Downtown Core         44880
Tech Park             22440
Residential West      18360
------------------------------------------------------------

Feature: 'road_id'
• Unique Values: 94
• Missing Values: 0 (0.00%)
• Top Value Distribution:
road_id
RD_332    6120
RD_489    4080
RD_147    4080
RD_100    4080
RD_150    4080
--

In [6]:
# =========================
# CAT COLUMNS VALUES CHECK
# =========================

cat_cols = df.select_dtypes(include= 'str').columns

for col in cat_cols:
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-' * 100)

record_id
204000
<ArrowStringArray>
['REC_0000001', 'REC_0000002', 'REC_0000003', 'REC_0000004', 'REC_0000005',
 'REC_0000006', 'REC_0000007', 'REC_0000008', 'REC_0000009', 'REC_0000010',
 ...
 'REC_0203991', 'REC_0203992', 'REC_0203993', 'REC_0203994', 'REC_0203995',
 'REC_0203996', 'REC_0203997', 'REC_0203998', 'REC_0203999', 'REC_0204000']
Length: 204000, dtype: str
----------------------------------------------------------------------------------------------------
city_zone
6
<ArrowStringArray>
['Financial District',    'Industrial East',     'Suburban North',
      'Downtown Core',          'Tech Park',   'Residential West']
Length: 6, dtype: str
----------------------------------------------------------------------------------------------------
road_id
94
<ArrowStringArray>
['RD_206', 'RD_869', 'RD_158', 'RD_230', 'RD_439', 'RD_489', 'RD_555',
 'RD_479', 'RD_692', 'RD_147', 'RD_868', 'RD_842', 'RD_798', 'RD_430',
 'RD_458', 'RD_332', 'RD_212', 'RD_515', 'RD_851', 'RD_246', 'RD_25

## Data Cleaning Plan

---

> Feature Elimination (Dropping Unnecessary Variables)
  * **`timestamp`**: Drop the raw and will retain all pre-extracted temporal columns instead.
  * **`record_id`**: Drop completely. It is a unique row identifier with zero predictive signal and would cause model overfitting.
  * **`road_id` & `intersection_id`**: Drop. High-cardinality spatial identifiers (94 roads and 100 intersections) would force tree models to memorize specific street names rather than learning generalizable urban mobility physics. Macro spatial zoning (`city_zone`, `road_type`) will be retained instead.

---

> Handling Extreme Numerical Skewness
  Our numerical check reveals heavy right-skewness in operational metrics:
  * **`average_wait_time`** (Skewness: 6.63, Max: 1,174 mins vs. Median: 1.9 mins)
  * **`emission_estimate`** (Skewness: 2.63, Max: 9,717g vs. Median: 74g)
  * **`queue_length`** (Skewness: 6.35, Max: 10,111m vs. Median: 19m)
  * *Action:* Apply logarithmic scaling (`np.log1p`) to normalize target distributions and prevent extreme anomaly spikes from destabilizing regressor convergence.

---

## Data Cleaning

In [7]:
# ==========================================
# FEATURE ELIMINATION (DROPPING UNNECESSARY VARIABLES)
# ==========================================

# Define columns to drop
columns_to_drop = ['record_id', 'road_id', 'intersection_id', 'timestamp']

# Verify which columns actually exist in the dataframe before dropping
existing_drops = [col for col in columns_to_drop if col in df.columns]

# Drop the columns
df = df.drop(columns=existing_drops)

print("Feature elimination complete!")
print(f"Dropped columns: {existing_drops}")
print(f"Updated DataFrame Shape: {df.shape[0]:,} rows | {df.shape[1]:,} columns\n")
print("Remaining columns preview:")
print(list(df.columns))  # Show remaining columns

Feature elimination complete!
Dropped columns: ['record_id', 'road_id', 'intersection_id', 'timestamp']
Updated DataFrame Shape: 204,000 rows | 43 columns

Remaining columns preview:
['city_zone', 'latitude', 'longitude', 'road_type', 'lanes', 'speed_limit', 'vehicle_count', 'average_speed', 'heavy_vehicle_count', 'motorcycle_count', 'public_transport_count', 'traffic_density', 'queue_length', 'average_wait_time', 'hour', 'day_of_week', 'is_weekend', 'is_holiday', 'rush_hour', 'peak_period', 'temperature', 'rainfall_mm', 'visibility_km', 'air_quality_index', 'weather_condition', 'nearby_school', 'nearby_hospital', 'nearby_market', 'construction_activity', 'road_condition', 'parking_occupancy', 'public_event', 'accident_reported', 'traffic_camera_count', 'iot_sensor_health', 'signal_cycle_seconds', 'green_light_duration', 'emergency_vehicle_detected', 'traffic_flow_rate', 'congestion_score', 'emission_estimate', 'fuel_waste_estimate', 'congestion_level']


In [8]:
# ==========================================
# HANDLING EXTREME NUMERICAL SKEWNESS & DROPPING ORIGINALS
# ==========================================

# 1. Define columns to log-transform
cols_to_log = ['average_wait_time', 'emission_estimate', 'queue_length']

# 2. Apply log1p transformation and create new log columns
for col in cols_to_log:
    if col in df.columns:
        log_col_name = f'log_{col}'
        df[log_col_name] = np.log1p(df[col])
        print(f"Log-Transformed: '{col}' -> '{log_col_name}'")

# 3. Drop the original raw un-logged columns to prevent redundancy and multicollinearity
existing_originals_to_drop = [col for col in cols_to_log if col in df.columns]
df = df.drop(columns=existing_originals_to_drop)
print(f"Dropped original raw columns: {existing_originals_to_drop}")

print(f"\nUpdated DataFrame Shape: {df.shape[0]:,} rows | {df.shape[1]:,} columns\n")
print("Preview of newly created log-transformed columns:")
display(df[['log_average_wait_time', 'log_emission_estimate', 'log_queue_length']].head(3))

Log-Transformed: 'average_wait_time' -> 'log_average_wait_time'
Log-Transformed: 'emission_estimate' -> 'log_emission_estimate'
Log-Transformed: 'queue_length' -> 'log_queue_length'
Dropped original raw columns: ['average_wait_time', 'emission_estimate', 'queue_length']

Updated DataFrame Shape: 204,000 rows | 43 columns

Preview of newly created log-transformed columns:


,log_average_wait_time,log_emission_estimate,log_queue_length
0,0.955511,3.692373,2.727853
1,0.832909,3.613347,2.564949
2,0.262364,3.806885,1.252763


### Memory Footprint Optimization & Downcasting

To optimize performance for real-time smart city data streams, we executed a downcasting function across our numerical data types:

* **Integers (`int64` $\rightarrow$ `int32`):** Halves the memory overhead for discrete counters (like vehicle counts and lane counts) while supporting values up to $\pm 2.14 \times 10^9$.

* **Floats (`float64` $\rightarrow$ `float32`):** Reduces memory storage for continuous measurements (like speed, coordinates, and weather metrics) from 8 bytes to 4 bytes per value.

**Why this matters for the project:**

1. **Production Best Practice:** Smart-city IoT sensor telemetry generates massive volumes of streaming data. Explicit memory management is a hallmark of production-ready data engineering.

2. **Native ML Compatibility:** High-performance gradient boosting frameworks (like XGBoost and LightGBM) natively execute training calculations in 32-bit precision. Casting early prevents redundant under-the-hood data conversions during model fitting.

In [9]:
# ==========================================
# MEMORY FOOTPRINT OPTIMIZATION
# ==========================================

def optimize_memory(data):
    start_mem = data.memory_usage().sum() / 1024**2
    
    # Downcast integers
    int_cols = data.select_dtypes(include=['int64']).columns
    data[int_cols] = data[int_cols].astype('int32')
    
    # Downcast floats
    float_cols = data.select_dtypes(include=['float64']).columns
    data[float_cols] = data[float_cols].astype('float32')
    
    end_mem = data.memory_usage().sum() / 1024**2
    savings = ((start_mem - end_mem) / start_mem) * 100
    
    print(f"Memory optimization complete!")
    print(f"Memory usage decreased from {start_mem:.2f} MB to {end_mem:.2f} MB ({savings:.1f}% reduction)")
    
    return data

# Apply the optimization to the dataframe (before splitting X and y)
df = optimize_memory(df)

Memory optimization complete!
Memory usage decreased from 76.97 MB to 48.96 MB (36.4% reduction)


In [10]:
# =============================================
# SAVING A COPY OF CLEANED DATA FOR DEPLOYMENT
# =============================================

# df.to_parquet('cleaned_df.parquet', index= False)

### Model Evaluation Architecture & Lineup

To ensure rigorous scientific validation, our CityFlow platform tests a multi-tiered progression of models—ranging from interpretable linear baselines to advanced gradient-boosting ensembles.

#### Classification Target (`congestion_level`)
*Goal: Predict categorical traffic severity (`Low`, `Moderate`, `High`, `Severe`)*
* **Linear Baselines:** `Logistic Regression (Multinomial)`, `Ridge Classifier`
* **Distance/Kernel-Based:** `KNN Classifier`, `SVC (Support Vector Classifier)`
* **Tree-Based & Ensembles:** `Decision Tree Classifier`, `Random Forest Classifier`, `Extra Trees Classifier`
* **Advanced Booster (Champion):** `XGBoost Classifier`
* **Key Evaluation Metrics:** Macro F1-Score, Weighted F1-Score, and Multiclass ROC-AUC.

#### Regression Targets (`log_average_wait_time`, `log_emission_estimate`)
*Goal: Forecast continuous delay times and pollutant masses*
* **Raw & Regularized Linear:** `Linear Regression (OLS)`, `Ridge ($L_2$)`, `Lasso ($L_1$)`
* **Non-Linear Linear:** `Polynomial Features` + Ridge/Linear
* **Gradient-Optimized & Distance:** `SGD Regressor`, `KNN Regressor`
* **Tree-Based & Ensembles:** `Decision Tree Regressor`, `Random Forest Regressor`
* **Advanced Booster (Champion):** `XGBoost Regressor`
* **Key Evaluation Metrics:** Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and $R^2$ Score.

## Data Preprocessing for ML


> Target & Feature Matrix ($X, y$) Separation
  Isolate our variables to establish the training arrays:
  * **Classification Target ($y_{class}$):** `congestion_level` (4 tiers: Low, Moderate, High, Severe).
  * **Regression Targets ($y_{reg}$):** `average_wait_time` and `emission_estimate`
  * **Feature Matrix ($X$):** All remaining clean categorical and numerical columns, stripped of target variables and dropped IDs.

### Split Data into Input and Output

In [11]:
# =============================================
# Define columns to exclude from feature matrix X (Targets + Leakage/Redundancies)
# =============================================

columns_to_exclude = [
    'congestion_level',         # Classification target
    'log_average_wait_time',    # Log regression target 1
    'log_emission_estimate',    # Log regression target 2
    'congestion_score',         # Excluded to prevent data leakage
    'fuel_waste_estimate'       # Excluded to prevent redundancy
]

# 3. Define Feature Matrix X and Target Arrays (y)
X = df.drop(columns=[col for col in columns_to_exclude if col in df.columns])

y_class = df['congestion_level']
y_reg_wait = df['log_average_wait_time']
y_reg_emission = df['log_emission_estimate']

# 4. Perform stratified train/test split (stratifying on classification target ensures balanced class ratios)
# We will split 80% train, 20% test with a fixed random seed for reproducibility
X_train, X_test, \
y_class_train, y_class_test, \
y_wait_train, y_wait_test, \
y_emit_train, y_emit_test = train_test_split(
    X, y_class, y_reg_wait, y_reg_emission,
    test_size=0.20,
    random_state=42,
    stratify=y_class
)

print(f"\nData split complete!")
print(f"Training Set Shape (X_train): {X_train.shape[0]:,} rows | {X_train.shape[1]:,} features")
print(f"Testing Set Shape  (X_test) : {X_test.shape[0]:,} rows")


Data split complete!
Training Set Shape (X_train): 163,200 rows | 38 features
Testing Set Shape  (X_test) : 40,800 rows


### Feature Matrix Exclusion Rationale (`columns_to_exclude`)

To maintain absolute structural integrity and prevent algorithmic degradation during training, five columns were explicitly excluded when defining the primary feature matrix $X$:

| Excluded Column | Category / Role | Core Engineering Reason for Exclusion |
| :--- | :--- | :--- |
| **`congestion_level`** | Multi-Class Target | **Primary Classification Output:** This is the categorical target array ($y_{class}$) representing discrete traffic severity tiers (`Low`, `Moderate`, `High`, `Severe`). |
| **`log_average_wait_time`** | Continuous Target | **Primary Regression Output 1:** The log-transformed wait duration target ($y_{wait}$) assigned to its dedicated regression pipeline. |
| **`log_emission_estimate`** | Continuous Target | **Primary Regression Output 2:** The log-transformed ecological impact target ($y_{emit}$) assigned to its dedicated regression pipeline. |
| **`congestion_score`** | Intermediate Metric | **Data Leakage Prevention:** `congestion_score` is a downstream mathematical composite directly reflecting real-time traffic severity. Including it as an input feature would give the model the "answer sheet" during training, destroying its ability to generalize on raw environmental inputs. |
| **`fuel_waste_estimate`** | Derived Metric | **Redundancy & Collinearity Removal:** `fuel_waste_estimate` exhibits near-perfect colinearity with `log_emission_estimate`. Retaining it as an input would introduce severe multicollinearity without adding distinct signal. |

**Summary:** Keeping these target and leakage features strictly firewalled within the master DataFrame (`df`) allowed us to perform complete statistical EDA, while dropping them from $X$ ensures our champion models learn purely from true, non-leaking operational inputs (e.g., vehicle volume, queue length, speed, and time metrics).

### Numerical Pipeline

In [12]:
# ===========================
# NUMERICAL COLUMNS PIPELINE
# ===========================

# 1. Automatically extract all numerical features from X_train
numerical_cols = X_train.select_dtypes(include=['number']).columns.tolist()

print(f"Identified {len(numerical_cols)} numerical features for preprocessing.")

# 2. Construct the Robust Numerical Pipeline
numerical_transformer = Pipeline(steps=[
    # Imputer: Fills any edge-case missing values with the median to prevent model crashes
    ('imputer', SimpleImputer(strategy='median')),
    
    # Scaler: Centers features (mean=0, variance=1)
    # CRITICAL: Mandatory for KNN, SVR/SVC, SGD, Ridge, and Lasso convergence/regularization stability.
    ('scaler', StandardScaler())
])

print("Numerical pipeline successfully constructed!")

Identified 32 numerical features for preprocessing.
Numerical pipeline successfully constructed!


> `SimpleImputer(strategy='median')`: Acts as an insurance policy. If the Streamlit web app or future data streams receive an unexpected missing value (NaN), the median fallback prevents the model from crashing.

> `StandardScaler()`: Normalizes features to a standard normal distribution. While tree-based models (like XGBoost) don't require scaling, building this scaling step into our pipeline is critical if we want to benchmark linear models, Support Vector Regressors (SVR), or Neural Networks (MLPs) fairly during our final model comparison.

In [13]:
numerical_transformer

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account fo

### Categorical Pipeline

In [14]:
# ==========================================
# CATEGORICAL COLUMNS PIPELINE
# ==========================================

# 1. Automatically identify all categorical (str, category, or boolean) features in X_train
categorical_cols = X_train.select_dtypes(include=['str', 'category', 'bool']).columns.tolist()

print(f"Identified {len(categorical_cols)} categorical features for encoding.")

# 2. Construct the Categorical Pipeline
categorical_transformer = Pipeline(steps=[
    # Imputer: Fills any missing text entries with a safe placeholder string
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    
    # One-Hot Encoder: Converts categorical labels into binary dummy columns (0 or 1)
    # CRITICAL: handle_unknown='ignore' to prevent pipeline from crashing if a brand new 
    # category shows up in our test set or live Streamlit app deployment!
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

print("Categorical pipeline successfully constructed!")

Identified 6 categorical features for encoding.
Categorical pipeline successfully constructed!


> When deploying the trained model into the interactive Streamlit app, users might input a combination or category that wasn't heavily present in our static training split. Setting `handle_unknown='ignore'` tells the encoder to safely encode unseen categories as all zeros instead of raising a fatal Python exception, ensuring industrial-grade fault tolerance.

In [15]:
categorical_transformer

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'constant'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",'missing'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto ac

### Pre-Flight Checks (Before Assembling)

> We must verify No Target or Leakage Columns Slipped Into `X_train` and confirm that targets (`congestion_level`, `log targets`) and leakage variables (`congestion_score`, `fuel_waste_estimate`) are completely dropped from `X_train`

In [16]:
print(f"Total columns in X_train: {X_train.shape[1]}")
print(f"Covered by pipelines: {len(numerical_cols) + len(categorical_cols)}")

Total columns in X_train: 38
Covered by pipelines: 38


### Master ColumnTransformer & Pipelines

In [17]:
# ==========================================
# MASTER COLUMN TRANSFORMER & PREPROCESSOR
# ==========================================

# 1. Standard Preprocessor (Used for XGBoost, Random Forest, KNN, Ridge, Lasso, SGD, Trees)
standard_preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'  # Safely drops any unlisted columns
)

# 2. Polynomial Preprocessor (Used exclusively for Polynomial + Ridge)
# Note: We apply polynomial features to numerical columns, while still one-hot encoding categorical columns

poly_numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('poly', PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)),
    ('scaler', StandardScaler())
])

polynomial_preprocessor = ColumnTransformer(
    transformers=[
        ('poly_num', poly_numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

print("Master ColumnTransformers successfully built!")
print(f"- Standard Preprocessor handles {len(numerical_cols)} numerical & {len(categorical_cols)} categorical features.")
print(f"- Polynomial Preprocessor ready for degree-2 interaction expansion.")

Master ColumnTransformers successfully built!
- Standard Preprocessor handles 32 numerical & 6 categorical features.
- Polynomial Preprocessor ready for degree-2 interaction expansion.


> Setting `remainder='drop'` acts as an architectural safety net. If a stray metadata column (like an ID string, a raw timestamp, or an unwanted text field) accidentally sneaks into `X_train` in the future, `drop` safely ignores it rather than crashing our training loop or poisoning the model input.

In [18]:
standard_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

> For the `Polynomial Regression` model, we construct a dedicated pipeline that injects `PolynomialFeatures` right before the scaler

> We also decided to use Polynominal features only with `Ridge`, because using it with `LinearRegression` will cause a significant expansion of features = multicollinearity = the model will severely overfit = training error might look fine, but test error will violently explode.

> `Ridge Regression` adds an `L2 penalty` ($\alpha$) to the `loss function`. This penalty forces the model to keep coefficient weights small and stable, effectively taming the chaos of polynomial expansion and preventing overfitting.

In [19]:
polynomial_preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('poly_num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_n

## Model Evaluation Metrics & Validation Strategy

To ensure rigorous, production-grade validation for our multi-target model, we established specific primary metrics and cross-validation techniques tailored to our regression and classification targets.

#### 1. Regression Targets (`log_average_wait_time`, `log_emission_estimate`)
* **Metrics Tracked:** Root Mean Squared Error (RMSE), Mean Absolute Error (MAE), and $R^2$ Score.
* **Primary Metric:** **RMSE (Root Mean Squared Error)**
* **Why:** RMSE squares errors before averaging them, heavily penalizing large prediction mistakes. In smart city traffic management, failing to predict a massive gridlock bottleneck by 30 minutes is far more critical than missing a minor delay by 2 minutes. RMSE ensures our models prioritize avoiding catastrophic errors.

RMSE is similar to MAE, but it squares the errors before averaging them, then takes the square root.

Squaring the errors changes everything. If the model misses a minor traffic delay by 2 minutes, $2^2 = 4$. But if model completely misses a massive gridlock bottleneck by 30 minutes, $30^2 = 900$.

Because RMSE squares large errors, it heavily penalizes massive mistakes. In smart city traffic management, failing to catch a catastrophic gridlock bottleneck is infinitely worse than being off by a minute on a normal street. RMSE forces our models to learn how to avoid disastrously wrong predictions.

#### 2. Classification Target (`congestion_level` — Multiclass)
* **Metrics Tracked:** Macro F1-Score, Weighted F1-Score, Accuracy, and Multiclass ROC-AUC.
* **Primary Metric:** **Macro F1-Score**
* **Why:** Traffic severity datasets are inherently imbalanced (rare "Severe" bottlenecks vs. frequent "Low" traffic). While accuracy can be misleading (a lazy model predicting "Low" always gets high accuracy), **Macro F1-Score** harmonizes Precision and Recall across all classes equally, guaranteeing robust performance even on rare, high-stakes severe traffic events.

#### 3. Cross-Validation Strategy
* **Method:** 5-Fold Cross-Validation (`KFold` for regression, `StratifiedKFold` for classification).
* **Why:** Evaluating models on a single random train/test split can introduce sampling bias. 5-fold cross-validation trains and tests models across 5 distinct data slices to ensure stable, statistically sound performance scores. Furthermore, `StratifiedKFold` ensures every fold preserves the exact class proportion ratio, preventing biased training splits for rare congestion categories.

## Regression Models Evaluation

### Regression Model Pramaters Strategy

To maximize predictive power while ensuring rigorous generalization and zero overfitting across our 38 feature dimensions, we fine-tuned our model using best practices:

1. **Regularized Linear Models (Ridge & Polynomial + Ridge):** 
   * *Adjustment:* Increased `alpha` to `10.0` (and `50.0` for Polynomial + Ridge).
   * *Why:* Polynomial expansion and correlated traffic features create severe multicollinearity (multiple features with high correlation causing redundancy). Higher regularization penalties keep coefficient weights strictly controlled, preventing the model from overfitting to training noise.
2. **Lasso ($L_1$):** 
   * *Adjustment:* Lowered `alpha` to `0.001` and increased `max_iter` to `5000`.
   * *Why:* A milder penalty retains critical non-linear traffic predictors from being zeroed out, while higher iterations ensure full mathematical convergence.
3. **SGD Regressor:** 
   * *Adjustment:* Adjusted convergence tolerances (`max_iter=2000`, `tol=1e-4`).
   * *Why:* Ensures gradient descent fully stabilizes without premature early stopping on scaled data matrices.
4. **KNN Regressor:** 
   * *Adjustment:* Increased neighbors to `n_neighbors=7` with `weights='distance'`.
   * *Why:* Smooths out localized spatial/temporal noise in urban intersections, while distance-weighting ensures closer neighbor nodes have a stronger vote on wait times.
5. **Decision Tree:** 
   * *Adjustment:* Constrained `max_depth=10` and `min_samples_split=20`.
   * *Why:* Unconstrained trees memorize training splits and overfit. Depth limits and split thresholds force the model to learn generalizable traffic rules.
6. **Random Forest:** 
   * *Adjustment:* Configured `n_estimators=150`, `max_depth=15`, `max_features='sqrt'`, and `n_jobs=-1`.
   * *Why:* Restricting features per split (`sqrt`) reduces tree correlation, making the ensemble significantly more robust. Parallelizing across all CPU cores (`n_jobs=-1`) ensures instant training times.
7. **XGBoost:** 
   * *Adjustment:* Lowered `learning_rate=0.05`, increased trees to `150`, and added row/column subsampling (`0.8`).
   * *Why:* A slower learning rate with more estimators produces a stable gradient booster. Subsampling 80% of rows and columns per tree introduces beneficial regularization to prevent overfitting.

In [20]:
# ==========================================
# 5-FOLD CV REGRESSION LOOP
# ==========================================

# 1. Define the 9 regression model pipelines (with optimized parameters)
regression_models = {
    "Linear Regression": Pipeline([('pre', standard_preprocessor), ('reg', LinearRegression())]),
    "Ridge L2": Pipeline([('pre', standard_preprocessor), ('reg', Ridge(alpha=10.0))]),
    "Lasso L1": Pipeline([('pre', standard_preprocessor), ('reg', Lasso(alpha=0.001, max_iter=5000, random_state=42))]),
    "Polynomial + Ridge": Pipeline([('pre', polynomial_preprocessor), ('reg', Ridge(alpha=50.0))]),
    "SGD Regressor": Pipeline([('pre', standard_preprocessor), ('reg', SGDRegressor(max_iter=2000, tol=1e-4, random_state=42))]),
    "KNN Regressor": Pipeline([('pre', standard_preprocessor), ('reg', KNeighborsRegressor(n_neighbors=7, weights='distance'))]),
    "Decision Tree": Pipeline([('pre', standard_preprocessor), ('reg', DecisionTreeRegressor(max_depth=10, min_samples_split=20, random_state=42))]),
    "Random Forest": Pipeline([('pre', standard_preprocessor), ('reg', RandomForestRegressor(n_estimators=150, max_depth=15, max_features='sqrt', random_state=42, n_jobs=-1))]),
    "XGBoost": Pipeline([('pre', standard_preprocessor), ('reg', XGBRegressor(n_estimators=150, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1))])
}

# 2. Dictionary of regression training targets
regression_targets = {
    "log_average_wait_time": y_wait_train.copy(),
    "log_emission_estimate": y_emit_train.copy()
}

# 3. Set up 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
X_train_data = X_train.copy()

# 4. Execute evaluation loop for each regression target
for target_name, y_target in regression_targets.items():
    cv_results = []
    print(f"\n==================================================")
    print(f"Running 5-Fold CV for Target: '{target_name}'")
    print(f"==================================================\n")
    
    for name, pipeline in regression_models.items():
        fold_rmse, fold_mae, fold_r2 = [], [], []
        
        for train_idx, val_idx in kf.split(X_train_data):
            X_fold_train, X_fold_val = X_train_data.iloc[train_idx], X_train_data.iloc[val_idx]
            y_fold_train, y_fold_val = y_target.iloc[train_idx], y_target.iloc[val_idx]

            # Fit and predict
            pipeline.fit(X_fold_train, y_fold_train)
            y_pred = pipeline.predict(X_fold_val)

            # Store multiclass metrics
            fold_rmse.append(np.sqrt(mean_squared_error(y_fold_val, y_pred)))
            fold_mae.append(mean_absolute_error(y_fold_val, y_pred))
            fold_r2.append(r2_score(y_fold_val, y_pred))
            
        cv_results.append({
            "Model": name,
            "Mean RMSE ↓ (Lower is Better)": np.mean(fold_rmse),
            "Std RMSE ± (Stability)": np.std(fold_rmse),
            "Mean MAE ↓ (Lower is Better)": np.mean(fold_mae),
            "Mean R² Score ↑ (Higher is Better)": np.mean(fold_r2)
        })
        print(f"Completed: {name:<20} | RMSE: {np.mean(fold_rmse):.4f} | MAE: {np.mean(fold_mae):.4f} | R²: {np.mean(fold_r2):.4f}")

    # Build and sort leaderboard DataFrame using the primary error metric column
    leaderboard_df = pd.DataFrame(cv_results).sort_values(by="Mean RMSE ↓ (Lower is Better)", ascending=True).reset_index(drop=True)
    
    print(f"\nLEADERBOARD FOR: {target_name}")
    
    # Format all numeric score columns as percentages
    styled_leaderboard = leaderboard_df.style.format({
        "Mean RMSE ↓ (Lower is Better)": lambda x: f"{x * 100:.2f}%",
        "Std RMSE ± (Stability)": lambda x: f"{x * 100:.2f}%",
        "Mean MAE ↓ (Lower is Better)": lambda x: f"{x * 100:.2f}%",
        "Mean R² Score ↑ (Higher is Better)": lambda x: f"{x * 100:.2f}%"
    })
    
    display(styled_leaderboard)


Running 5-Fold CV for Target: 'log_average_wait_time'

Completed: Linear Regression    | RMSE: 0.1689 | MAE: 0.1306 | R²: 0.9841
Completed: Ridge L2             | RMSE: 0.1689 | MAE: 0.1306 | R²: 0.9841
Completed: Lasso L1             | RMSE: 0.1705 | MAE: 0.1311 | R²: 0.9838
Completed: Polynomial + Ridge   | RMSE: 0.1398 | MAE: 0.1107 | R²: 0.9891
Completed: SGD Regressor        | RMSE: 0.1694 | MAE: 0.1309 | R²: 0.9840
Completed: KNN Regressor        | RMSE: 0.3082 | MAE: 0.2287 | R²: 0.9469
Completed: Decision Tree        | RMSE: 0.0842 | MAE: 0.0657 | R²: 0.9960
Completed: Random Forest        | RMSE: 0.1702 | MAE: 0.1302 | R²: 0.9838
Completed: XGBoost              | RMSE: 0.0827 | MAE: 0.0650 | R²: 0.9962

LEADERBOARD FOR: log_average_wait_time


,Model,Mean RMSE ↓ (Lower is Better),Std RMSE ± (Stability),Mean MAE ↓ (Lower is Better),Mean R² Score ↑ (Higher is Better)
0,XGBoost,8.27%,0.02%,6.50%,99.62%
1,Decision Tree,8.42%,0.03%,6.57%,99.60%
2,Polynomial + Ridge,13.98%,0.09%,11.07%,98.91%
3,Linear Regression,16.89%,0.06%,13.06%,98.41%
4,Ridge L2,16.89%,0.06%,13.06%,98.41%
5,SGD Regressor,16.94%,0.05%,13.09%,98.40%
6,Random Forest,17.02%,0.16%,13.02%,98.38%
7,Lasso L1,17.05%,0.07%,13.11%,98.38%
8,KNN Regressor,30.82%,0.22%,22.87%,94.69%



Running 5-Fold CV for Target: 'log_emission_estimate'

Completed: Linear Regression    | RMSE: 0.1364 | MAE: 0.1001 | R²: 0.9880
Completed: Ridge L2             | RMSE: 0.1364 | MAE: 0.1001 | R²: 0.9880
Completed: Lasso L1             | RMSE: 0.1375 | MAE: 0.1007 | R²: 0.9878
Completed: Polynomial + Ridge   | RMSE: 0.0589 | MAE: 0.0437 | R²: 0.9978
Completed: SGD Regressor        | RMSE: 0.1373 | MAE: 0.1010 | R²: 0.9878
Completed: KNN Regressor        | RMSE: 0.1424 | MAE: 0.0904 | R²: 0.9869
Completed: Decision Tree        | RMSE: 0.0298 | MAE: 0.0174 | R²: 0.9994
Completed: Random Forest        | RMSE: 0.0336 | MAE: 0.0206 | R²: 0.9993
Completed: XGBoost              | RMSE: 0.0260 | MAE: 0.0140 | R²: 0.9996

LEADERBOARD FOR: log_emission_estimate


,Model,Mean RMSE ↓ (Lower is Better),Std RMSE ± (Stability),Mean MAE ↓ (Lower is Better),Mean R² Score ↑ (Higher is Better)
0,XGBoost,2.60%,0.04%,1.40%,99.96%
1,Decision Tree,2.98%,0.03%,1.74%,99.94%
2,Random Forest,3.36%,0.12%,2.06%,99.93%
3,Polynomial + Ridge,5.89%,0.02%,4.37%,99.78%
4,Linear Regression,13.64%,0.11%,10.01%,98.80%
5,Ridge L2,13.64%,0.11%,10.01%,98.80%
6,SGD Regressor,13.73%,0.10%,10.10%,98.78%
7,Lasso L1,13.75%,0.10%,10.07%,98.78%
8,KNN Regressor,14.24%,0.09%,9.04%,98.69%


### Single Metric vs. Train-Test Splits

You might notice that each model reports a single consolidated score per metric (e.g., Mean RMSE and Mean $R^2$) rather than separate Train and Test scores. Here is why it is like this in our design:

#### 1. Why is the Score a Single Number?

In 5-fold cross-validation, the dataset is split into 5 distinct slices. For each iteration, the model is trained on 80% of the data and evaluated *exclusively* on the remaining 20% unseen validation fold. 

* **Out-of-Sample Focus:** We discard the training error for each fold and record only the validation error—because how a model performs on data it has *never seen before* is the true test of its intelligence.

* **Averaged Generalization:** The final score displayed on the leaderboard is the mathematical average (mean) of those 5 separate validation scores, accompanied by their standard deviation ($\pm$ std) to measure stability. 

#### 2. Which is Better? (Cross-Validated Validation Score vs. Train/Test Comparison)

* **Cross-Validated Validation Score (Our Approach) is Better for Model Selection:** Relying on out-of-sample cross-validation scores prevents choosing a model that merely memorized the training data (overfitting). It gives a bulletproof, statistically rigorous guarantee that the model will generalize well in production.

* **Train vs. Test Comparison is Strictly for Debugging:** Comparing training scores against test scores is only useful if we suspect overfitting (e.g., if a model has a training $R^2$ of 1.0 but a validation $R^2$ of 0.5). However, when the XGBoost model achieves a stellar, tightly clustered validation $R^2$ of 0.996 with negligible standard deviation ($\pm0.0004$), it proves the pipelines are generalizing cleanly without overfitting—making the single cross-validated validation score the definitive metric for the leaderboard.

#### 3. Why Single-Metric (RMSE) Primary Optimization?

When evaluating machine learning pipelines for production systems, it is tempting to try and balance or average multiple competing metrics into a single score. However, for this project, **sorting and selecting models strictly based on our primary metric (RMSE)** is operationally the correct approach for several key reasons:

1. **Alignment with Real-World Domain Stakes:**

   * Different metrics measure error in fundamentally different ways. Mean Absolute Error (MAE) treats every mistake linearly, whereas Root Mean Squared Error (RMSE) exponentially squares errors before averaging.

   * In traffic management, being off by 2 minutes on a normal street is a minor annoyance, but failing to predict a massive gridlock bottleneck by 30 minutes is catastrophic. RMSE aligns our machine learning objective directly with our business tolerance for severe errors, forcing models to prioritize avoiding gridlock miscalculations.

2. **Avoiding Arbitrary Hybrid Formulas:**

   * Blending multiple metrics together creates an uninterpretable mathematical hybrid. A composite score lacks clear physical meaning and dilutes the specific penalty weight required for high-stakes urban forecasting.

3. **Unanimous Consensus Validated the Choice:**
 
   * Fortunately, our rigorous evaluation proved that compromise wasn't necessary. XGBoost didn't just win on RMSE—it universally dominated across every single evaluation axis (lowest RMSE, lowest MAE, and highest $R^2$). Sticking to our primary metric gave us a clean, indisputable champion grounded in first-principles data science.

## Classification Models Evaluation

### Classification Model Paramaters Optimization Strategy

To ensure robust multi-class classification performance across imbalanced urban traffic states (such as rare severe bottlenecks), we fine-tuned our classifiers using best practices:

1. **Logistic Regression & SGD Classifier:** 
   * *Adjustment:* Increased inverse regularization `C` (to `2.0`) and enabled `class_weight='balanced'`.
   * *Why:* Balanced weighting automatically penalizes misclassifications on rare severe traffic classes, ensuring the model doesn't ignore critical gridlock events in favor of dominant low-traffic categories.
2. **KNN Classifier:** 
   * *Adjustment:* Increased neighbors to `n_neighbors=9` with distance weighting and switched to `manhattan` distance.
   * *Why:* Manhattan distance outperforms Euclidean distance in high-dimensional tabular feature spaces (such as our 38 feature dimensions) by mitigating the curse of dimensionality.
3. **Decision Tree & Random Forest:** 
   * *Adjustment:* Expanded max depth (`12` and `18`), increased estimators (`200`), and applied `class_weight='balanced_subsample'`.
   * *Why:* Dynamic bootstrapping with balanced subsampling recalculates class weights inside every tree split, providing high-resolution decision boundaries for complex multi-class congestion patterns without overfitting.
4. **XGBoost Classifier:** 
   * *Adjustment:* Increased estimators to `200`, lowered learning rate to `0.05`, and added row/column subsampling (`0.8`) with explicit multi-class log loss tracking (`eval_metric='mlogloss'`).
   * *Why:* Controlled gradient boosting with subsampling regularizes tree growth, yielding high stability and robust Macro F1 scores across all traffic severity tiers.
5. **Support Vector Machine (SVM):** 
   * *Adjustment:* Increased `C=5.0`, enabled `class_weight='balanced'`, and turned on `probability=True`.
   * *Why:* A higher penalty parameter allows the RBF kernel to draw tight, precise decision boundaries around intricate multi-class traffic clusters.

In [21]:
# ==========================================
# 5-FOLD STRATIFIED CV CLASSIFICATION LOOP
# ==========================================

# 1. Define the 8 classification model pipelines (with optimized tabular parameters)
classification_models = {
    "Logistic Regression": Pipeline([('pre', standard_preprocessor), ('clf', LogisticRegression(C=2.0, class_weight='balanced', max_iter=2000, random_state=42, n_jobs=-1))]),
    "SGD Classifier": Pipeline([('pre', standard_preprocessor), ('clf', SGDClassifier(alpha=1e-5, penalty='l2', class_weight='balanced', max_iter=2000, tol=1e-4, random_state=42, loss='log_loss'))]),
    "KNN Classifier": Pipeline([('pre', standard_preprocessor), ('clf', KNeighborsClassifier(n_neighbors=9, weights='distance', metric='manhattan', n_jobs=-1))]),
    "Decision Tree": Pipeline([('pre', standard_preprocessor), ('clf', DecisionTreeClassifier(max_depth=12, min_samples_split=15, class_weight='balanced', random_state=42))]),
    "Random Forest": Pipeline([('pre', standard_preprocessor), ('clf', RandomForestClassifier(n_estimators=200, max_depth=18, max_features='sqrt', class_weight='balanced_subsample', random_state=42, n_jobs=-1))]),
    "XGBoost Classifier": Pipeline([('pre', standard_preprocessor), ('clf', XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss', random_state=42, n_jobs=-1))]),
    "Gaussian Naive Bayes": Pipeline([('pre', standard_preprocessor), ('clf', GaussianNB())]),
    "Support Vector Machine": Pipeline([('pre', standard_preprocessor), ('clf', SVC(C=5.0, gamma='scale', class_weight='balanced', probability=True, random_state=42))])
}

# 2. Encode string classification targets into integers ([0, 1, 2, 3...]) for XGBoost compatibility
encoder = LabelEncoder()
y_class_encoded = encoder.fit_transform(y_class_train)

print("Target Class Mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

# 3. Set up 5-Fold Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_train_data = X_train.copy()
y_class_target = y_class_encoded

cv_class_results = []
print(f"\n==================================================")
print(f"Running 5-Fold Stratified CV for Classification Target: 'congestion_level'")
print(f"==================================================\n")

for name, pipeline in classification_models.items():
    fold_macro_f1 = []
    fold_weighted_f1 = []
    fold_accuracy = []
    
    for train_idx, val_idx in skf.split(X_train_data, y_class_target):
        X_fold_train, X_fold_val = X_train_data.iloc[train_idx], X_train_data.iloc[val_idx]
        y_fold_train, y_fold_val = y_class_target[train_idx], y_class_target[val_idx]
        
        # Fit and predict
        pipeline.fit(X_fold_train, y_fold_train)
        y_pred = pipeline.predict(X_fold_val)
        
        # Store multiclass metrics
        fold_macro_f1.append(f1_score(y_fold_val, y_pred, average='macro'))
        fold_weighted_f1.append(f1_score(y_fold_val, y_pred, average='weighted'))
        fold_accuracy.append(accuracy_score(y_fold_val, y_pred))
        
    cv_class_results.append({
        "Model": name,
        "Mean Macro F1-Score ↑ (Primary Metric)": np.mean(fold_macro_f1),
        "Std Macro F1 ± (Stability)": np.std(fold_macro_f1),
        "Mean Weighted F1 ↑ (Higher is Better)": np.mean(fold_weighted_f1),
        "Mean Accuracy ↑ (Higher is Better)": np.mean(fold_accuracy)
    })
    print(f"Completed: {name:<22} | Mean Macro F1: {np.mean(fold_macro_f1):.4f} | Mean Accuracy: {np.mean(fold_accuracy):.4f}")

# Build, sort (descending by highest Macro F1), and format leaderboard DataFrame
class_leaderboard_df = pd.DataFrame(cv_class_results).sort_values(by="Mean Macro F1-Score ↑ (Primary Metric)", ascending=False).reset_index(drop=True)

print(f"\nCLASSIFICATION MODEL LEADERBOARD (congestion_level):")

# Format all evaluation metrics as clean percentages
styled_class_leaderboard = class_leaderboard_df.style.format({
    "Mean Macro F1-Score ↑ (Primary Metric)": lambda x: f"{x * 100:.2f}%",
    "Std Macro F1 ± (Stability)": lambda x: f"{x * 100:.2f}%",
    "Mean Weighted F1 ↑ (Higher is Better)": lambda x: f"{x * 100:.2f}%",
    "Mean Accuracy ↑ (Higher is Better)": lambda x: f"{x * 100:.2f}%"
})

display(styled_class_leaderboard)

Target Class Mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Moderate': np.int64(2), 'Severe': np.int64(3)}

Running 5-Fold Stratified CV for Classification Target: 'congestion_level'



c:\Users\a\.conda\envs\env1\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\a\.conda\envs\env1\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\a\.conda\envs\env1\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\Users\a\.conda\envs\env1\Lib\site-packages\sklearn\linear_model\_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspec

Completed: Logistic Regression    | Mean Macro F1: 0.9728 | Mean Accuracy: 0.9856
Completed: SGD Classifier         | Mean Macro F1: 0.9386 | Mean Accuracy: 0.9740
Completed: KNN Classifier         | Mean Macro F1: 0.9187 | Mean Accuracy: 0.9609
Completed: Decision Tree          | Mean Macro F1: 0.9750 | Mean Accuracy: 0.9871
Completed: Random Forest          | Mean Macro F1: 0.9730 | Mean Accuracy: 0.9865
Completed: XGBoost Classifier     | Mean Macro F1: 0.9884 | Mean Accuracy: 0.9943
Completed: Gaussian Naive Bayes   | Mean Macro F1: 0.8435 | Mean Accuracy: 0.9154


c:\Users\a\.conda\envs\env1\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\a\.conda\envs\env1\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\a\.conda\envs\env1\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\a\.conda\envs\env1\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in versio

Completed: Support Vector Machine | Mean Macro F1: 0.9584 | Mean Accuracy: 0.9778

CLASSIFICATION MODEL LEADERBOARD (congestion_level):


,Model,Mean Macro F1-Score ↑ (Primary Metric),Std Macro F1 ± (Stability),Mean Weighted F1 ↑ (Higher is Better),Mean Accuracy ↑ (Higher is Better)
0,XGBoost Classifier,98.84%,0.05%,99.42%,99.43%
1,Decision Tree,97.50%,0.23%,98.73%,98.71%
2,Random Forest,97.30%,0.11%,98.65%,98.65%
3,Logistic Regression,97.28%,0.11%,98.59%,98.56%
4,Support Vector Machine,95.84%,0.12%,97.83%,97.78%
5,SGD Classifier,93.86%,0.26%,97.43%,97.40%
6,KNN Classifier,91.87%,0.14%,95.99%,96.09%
7,Gaussian Naive Bayes,84.35%,0.29%,92.28%,91.54%


### Cross-Validation Strategy: K-Fold vs. Stratified K-Fold

To ensure robust, unbiased out-of-sample evaluation across our multi-target machine learning pipeline, we tailored our cross-validation strategy to the specific statistical nature of each target variable:

* **Standard `KFold` (for Regressors):** Because regression targets are continuous floating-point values, this was used for continuous numerical targets (`log_average_wait_time` and `log_emission_estimate`). It splits the dataset into $K$ equal sequential subsets, systematically rotating the validation fold across iterations. This ensures every data point gets an equal chance to validate model performance across continuous numeric distributions.

* **`StratifiedKFold` (for Classifiers):** Used for the categorical target (`congestion_level`). Unlike standard K-Fold, stratification guarantees that **each fold maintains the exact same percentage-based class distribution** (e.g., proportion of Low, Moderate, High, and Severe traffic tiers) as the overall dataset. This prevents class imbalance bias and ensures rare traffic severity states are fairly represented in every training and validation loop, because traffic datasets often contain class imbalances (e.g., normal flow occurs far more frequently than severe gridlock).

## Final Model Selection Summary

Following rigorous 5-fold cross-validation (standardized K-Fold for regression and Stratified K-Fold for classification), **XGBoost** emerged as the undisputed, universal champion across all three target variables of our CityFlow intelligence platform:

1. **Regression Task 1 (`log_average_wait_time`):**
   * **Champion:** XGBoost Regressor
   * **Performance:** Mean RMSE of **8.27%** (0.0827) and an exceptional Mean $R^2$ of **99.62%**.
   * **Why it won:** Effectively captured complex non-linear bottlenecks while strictly penalizing gridlock prediction errors.

2. **Regression Task 2 (`log_emission_estimate`):**
   * **Champion:** XGBoost Regressor
   * **Performance:** Mean RMSE of **2.60%** (0.0260) and a near-perfect Mean $R^2$ of **99.96%**.
   * **Why it won:** Outstanding gradient boosting convergence on continuous ecological metrics with virtually zero residual variance.

3. **Multi-Class Classification Task (`congestion_level`):**
   * **Champion:** XGBoost Classifier
   * **Performance:** Mean Macro F1-Score of **98.84%** and Mean Accuracy of **99.43%** ($\pm0.05\%$ stability).
   * **Why it won:** Handled multi-class stratification flawlessly, balancing high precision and recall across all traffic severity tiers without sacrificing rare bottleneck detection.

**Conclusion:** XGBoost’s built-in regularization, handling of tabular feature interactions, and tree-ensemble efficiency made it the definitive production-grade architecture for our portfolio dashboard.

## Hyperparameter Tuning for Best Models

In [ ]:
# =======================================================================
# HYPERPARAMETER TUNING FOR XGBOOST REGRESSOR FOR 'log_average_wait_time'
# =======================================================================

# Define the parameter search space for regression
param_wt = {
    'reg__n_estimators': randint(100, 300),
    'reg__learning_rate': uniform(0.01, 0.1),       # 0.01 to 0.11
    'reg__max_depth': randint(4, 9),
    'reg__subsample': uniform(0.7, 0.3),            # 0.7 to 1.0
    'reg__colsample_bytree': uniform(0.7, 0.3),     # 0.7 to 1.0
    'reg__min_child_weight': randint(1, 5),
    'reg__gamma': uniform(0, 0.5)
}

# Construct pipeline with base XGBoost regressor
xgb_wt_pipeline = Pipeline([
    ('pre', standard_preprocessor), 
    ('reg', XGBRegressor(random_state=42, n_jobs=-1))
])

# Set up Randomized Search (using 3-fold CV for speed, optimizing for negative root mean squared error)
xgb_wt_search = RandomizedSearchCV(
    estimator=xgb_wt_pipeline,
    param_distributions=param_wt,
    n_iter=15,          # Tests 15 random combinations
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Run search on your wait time target
print("Tuning XGBoost Regressor for 'log_average_wait_time'...")
xgb_wt_search.fit(X_train, y_wait_train)

print(f"\nBest Parameters Found:")
for param, val in xgb_wt_search.best_params_.items():
    print(f"   {param}: {val}")

best_wt_regressor = xgb_wt_search.best_estimator_

Tuning XGBoost Regressor for 'log_average_wait_time'...
Fitting 3 folds for each of 15 candidates, totalling 45 fits

Best Parameters Found:
   reg__colsample_bytree: 0.9181815987569262
   reg__gamma: 0.1632703844029177
   reg__learning_rate: 0.06704439744053994
   reg__max_depth: 7
   reg__min_child_weight: 1
   reg__n_estimators: 289
   reg__subsample: 0.7975990992289792


In [23]:
# =======================================================================
# HYPERPARAMETER TUNING FOR XGBOOST REGRESSOR FOR 'log_emission_estimate'
# =======================================================================

# 1. Define the parameter search space for emissions regression
param_dist_emissions = {
    'reg__n_estimators': randint(100, 300),
    'reg__learning_rate': uniform(0.01, 0.1),       # 0.01 to 0.11
    'reg__max_depth': randint(4, 9),
    'reg__subsample': uniform(0.7, 0.3),            # 0.7 to 1.0
    'reg__colsample_bytree': uniform(0.7, 0.3),     # 0.7 to 1.0
    'reg__min_child_weight': randint(1, 5),
    'reg__gamma': uniform(0, 0.5)
}

# 2. Construct pipeline with base XGBoost regressor
xgb_emissions_pipeline = Pipeline([
    ('pre', standard_preprocessor), 
    ('reg', XGBRegressor(random_state=42, n_jobs=-1))
])

# 3. Set up Randomized Search optimizing for negative root mean squared error
xgb_emissions_search = RandomizedSearchCV(
    estimator=xgb_emissions_pipeline,
    param_distributions=param_dist_emissions,
    n_iter=15,          # Tests 15 random combinations
    scoring='neg_root_mean_squared_error',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 4. Run search on your emission estimate target
print("Tuning XGBoost Regressor for 'log_emission_estimate'...")
xgb_emissions_search.fit(X_train, y_emit_train)

print(f"\nBest Parameters Found for Emissions:")
for param, val in xgb_emissions_search.best_params_.items():
    print(f"   {param}: {val}")

best_xgb_emissions_regressor = xgb_emissions_search.best_estimator_

Tuning XGBoost Regressor for 'log_emission_estimate'...
Fitting 3 folds for each of 15 candidates, totalling 45 fits

Best Parameters Found for Emissions:
   reg__colsample_bytree: 0.8703100983459974
   reg__gamma: 0.01565664622777929
   reg__learning_rate: 0.09422847745949985
   reg__max_depth: 7
   reg__min_child_weight: 2
   reg__n_estimators: 289
   reg__subsample: 0.9779976597381381


In [24]:
# AGAIN FOR RANDOMIZED SEARCH (BECAUSE CLASSIFIER EVALUATION TOOK 100MIN)

# Encode string classification targets into integers ([0, 1, 2, 3...]) for XGBoost compatibility 
encoder = LabelEncoder()
y_class_encoded = encoder.fit_transform(y_class_train)

print("Target Class Mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

Target Class Mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Moderate': np.int64(2), 'Severe': np.int64(3)}


In [25]:
# =============================================
# HYPERPARAMETER TUNING FOR XGBOOST CLASSIFIER
# =============================================

# Encode string classification targets into integers ([0, 1, 2, 3...]) for XGBoost compatibility 
encoder = LabelEncoder()
y_class_encoded = encoder.fit_transform(y_class_train)

print("Target Class Mapping:", dict(zip(encoder.classes_, encoder.transform(encoder.classes_))))

# Define the parameter search space for classification
param_dist_clf = {
    'clf__n_estimators': randint(100, 300),
    'clf__learning_rate': uniform(0.01, 0.1),
    'clf__max_depth': randint(4, 9),
    'clf__subsample': uniform(0.7, 0.3),
    'clf__colsample_bytree': uniform(0.7, 0.3),
    'clf__min_child_weight': randint(1, 5),
    'clf__gamma': uniform(0, 0.5)
}

# Construct pipeline with base XGBoost classifier
xgb_clf_pipeline = Pipeline([
    ('pre', standard_preprocessor), 
    ('clf', XGBClassifier(eval_metric='mlogloss', random_state=42, n_jobs=-1))
])

# Set up Randomized Search optimizing for Macro F1
xgb_clf_search = RandomizedSearchCV(
    estimator=xgb_clf_pipeline,
    param_distributions=param_dist_clf,
    n_iter=15,
    scoring='f1_macro',
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# Run search on encoded categorical target
print("\nTuning XGBoost Classifier for 'congestion_level'...")
xgb_clf_search.fit(X_train, y_class_encoded)

print(f"\nBest Parameters Found:")
for param, val in xgb_clf_search.best_params_.items():
    print(f"   {param}: {val}")

best_xgb_classifier = xgb_clf_search.best_estimator_


Target Class Mapping: {'High': np.int64(0), 'Low': np.int64(1), 'Moderate': np.int64(2), 'Severe': np.int64(3)}

Tuning XGBoost Classifier for 'congestion_level'...
Fitting 3 folds for each of 15 candidates, totalling 45 fits

Best Parameters Found:
   clf__colsample_bytree: 0.8123620356542087
   clf__gamma: 0.4753571532049581
   clf__learning_rate: 0.0831993941811405
   clf__max_depth: 8
   clf__min_child_weight: 1
   clf__n_estimators: 202
   clf__subsample: 0.8337498258560773


### Hyperparameter Optimization Summary (`RandomizedSearchCV`)

To maximize predictive efficiency and prevent overfitting across our universal champion **XGBoost** architectures, we conducted 3-fold cross-validated randomized parameter searches (15 candidate iterations each) across the primary search space.

#### Optimal Parameter Configurations

| Target Variable / Task | Model Architecture | Key Hyperparameters | Architectural & Domain Justification |
| :--- | :--- | :--- | :--- |
| **`log_average_wait_time`** <br>*(Regression)* | `XGBRegressor` | • `n_estimators`: **289**<br>• `learning_rate`: **0.0670**<br>• `max_depth`: **7**<br>• `subsample`: **0.7976**<br>• `colsample_bytree`: **0.9182**<br>• `gamma`: **0.1633**<br>• `min_child_weight`: **1** | Deeper column sampling (`91.8%`) paired with moderate regularizing gamma (`0.163`) isolates non-linear queue bottlenecks without sacrificing feature breadth. |
| **`log_emission_estimate`** <br>*(Regression)* | `XGBRegressor` | • `n_estimators`: **289**<br>• `learning_rate`: **0.0942**<br>• `max_depth`: **7**<br>• `subsample`: **0.9780**<br>• `colsample_bytree`: **0.8703**<br>• `gamma`: **0.0157**<br>• `min_child_weight`: **2** | High row subsample coverage (`97.8%`) and faster learning rate (`0.094`) reflect the smooth, continuous physical nature of vehicle emission outputs. |
| **`congestion_level`** <br>*(Classification)* | `XGBClassifier` | • `n_estimators`: **202**<br>• `learning_rate`: **0.0832**<br>• `max_depth`: **8**<br>• `subsample`: **0.8337**<br>• `colsample_bytree`: **0.8124**<br>• `gamma`: **0.4754**<br>• `min_child_weight`: **1** | Stronger regularization (`gamma=0.475`) alongside deep tree partitioning (`max_depth=8`) cleanly resolves complex boundary decision splits across multi-class traffic tiers. |

**Key Takeaway:** The tuned hyperparameter profiles confirm robust, task-specific convergence—capturing intricate tabular feature interactions while maintaining strict variance control prior to final serialization and Streamlit deployment.

## Model Serialization

In [26]:
# =======================================
# FULL TRAINING & ARTIFACT SERIALIZATION
# =======================================

print("Initializing final champion pipelines with tuned hyperparameters...")

# 1. Wait Time Regressor Champion
final_xgb_wait = Pipeline([
    ('pre', standard_preprocessor),
    ('reg', XGBRegressor(
        colsample_bytree=0.9181815987569262,
        gamma=0.1632703844029177,
        learning_rate=0.06704439744053994,
        max_depth=7,
        min_child_weight=1,
        n_estimators=289,
        subsample=0.7975990992289792,
        random_state=42,
        n_jobs=-1
    ))
])

# 2. Emissions Estimate Regressor Champion
final_xgb_emissions = Pipeline([
    ('pre', standard_preprocessor),
    ('reg', XGBRegressor(
        colsample_bytree=0.8703100983459974,
        gamma=0.01565664622777929,
        learning_rate=0.09422847745949985,
        max_depth=7,
        min_child_weight=2,
        n_estimators=289,
        subsample=0.9779976597381381,
        random_state=42,
        n_jobs=-1
    ))
])

# 3. Congestion Level Classifier Champion
final_xgb_classifier = Pipeline([
    ('pre', standard_preprocessor),
    ('clf', XGBClassifier(
        colsample_bytree=0.8123620356542087,
        gamma=0.4753571532049581,
        learning_rate=0.0831993941811405,
        max_depth=8,
        min_child_weight=1,
        n_estimators=202,
        subsample=0.8337498258560773,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    ))
])

# 4. Fit each pipeline on its respective full training dataset
print("Training final models on full dataset batches...")
final_xgb_wait.fit(X_train, y_wait_train)
print("\n   [✓] Wait Time Regressor trained successfully.")

final_xgb_emissions.fit(X_train, y_emit_train)
print("   [✓] Emissions Regressor trained successfully.")

final_xgb_classifier.fit(X_train, y_class_encoded)
print("   [✓] Congestion Classifier trained successfully.")

# 5. Modular serialization to disk
print("\nSerializing artifacts via joblib...")
joblib.dump(final_xgb_wait, "model_xgb_wait_time.joblib")
joblib.dump(final_xgb_emissions, "model_xgb_emissions.joblib")
joblib.dump(final_xgb_classifier, "model_xgb_congestion_classifier.joblib")
joblib.dump(standard_preprocessor, "standard_preprocessor.joblib")
joblib.dump(encoder, "congestion_label_encoder.joblib")

print("All champion models, feature preprocessors, and label encoders successfully saved as .joblib files!")

print("All champion models, feature preprocessors, and label encoders successfully saved to disk!")

Initializing final champion pipelines with tuned hyperparameters...
Training final models on full dataset batches...

   [✓] Wait Time Regressor trained successfully.
   [✓] Emissions Regressor trained successfully.
   [✓] Congestion Classifier trained successfully.

Serializing artifacts via joblib...
All champion models, feature preprocessors, and label encoders successfully saved as .joblib files!
All champion models, feature preprocessors, and label encoders successfully saved to disk!


## Feature Importance

In [31]:
# ==============================
# FEATURE IMPORTANCE EXTRACTION
# ==============================

# 1. Extract feature importances from the trained classifier
importances = final_xgb_classifier.named_steps['clf'].feature_importances_

# 2. Get the transformed feature names directly from the preprocessor step
try:
    feature_names = final_xgb_classifier.named_steps['pre'].get_feature_names_out()
except AttributeError:
    # Fallback in case an older scikit-learn version is used
    feature_names = [f"feature_{i}" for i in range(len(importances))]

# 3. Create and save a clean importance dataframe
fi_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
joblib.dump(fi_df, 'feature_importances.joblib')

print(f"Feature importances saved successfully! ({len(feature_names)} features aligned)")

Feature importances saved successfully! (56 features aligned)


## Manual Inference Test

In [ ]:
# 1. Load modular artifacts
model_wait = joblib.load("model_xgb_wait_time.joblib")
model_emissions = joblib.load("model_xgb_emissions.joblib")
model_classifier = joblib.load("model_xgb_congestion_classifier.joblib")
encoder = joblib.load("congestion_label_encoder.joblib")

# 2. Grab a test sample
sample_row = X_test.sample(n=1)

print("\n--- RAW INPUT SAMPLE ---")
display(sample_row)

# 3. Generate raw predictions
pred_log_wait = model_wait.predict(sample_row)[0]
pred_log_emissions = model_emissions.predict(sample_row)[0]
pred_class_encoded = model_classifier.predict(sample_row)[0]

# 4. REVERSE THE LOG TRANSFORMATION using np.expm1()
predicted_wait_minutes = np.expm1(pred_log_wait)
predicted_emissions_kg = np.expm1(pred_log_emissions)
predicted_congestion_label = encoder.inverse_transform([pred_class_encoded])[0]

# 5. Display human-readable results
print("\n--- TESTING CITYFLOW PREDICTIONS ---")
print(f"Estimated Wait Time:      {predicted_wait_minutes:.1f} Minutes")
print(f"Estimated Emissions:      {predicted_emissions_kg:.1f} Kilograms (CO₂)")
print(f"Traffic Congestion Level: {predicted_congestion_label}")


--- RAW INPUT SAMPLE ---


,city_zone,latitude,longitude,road_type,lanes,speed_limit,vehicle_count,average_speed,heavy_vehicle_count,motorcycle_count,...,parking_occupancy,public_event,accident_reported,traffic_camera_count,iot_sensor_health,signal_cycle_seconds,green_light_duration,emergency_vehicle_detected,traffic_flow_rate,log_queue_length
117514,Downtown Core,40.762489,-73.92804,Collector,2,40,990,18.4,28,67,...,54.900002,0,0,2,Optimal,60,40,0,990.0,4.867535



--- TESTING CITYFLOW PREDICTIONS ---
Estimated Wait Time:      12.5 minutes
Estimated Emissions:      281.6 Kilograms (CO₂)
Traffic Congestion Level: Severe


## Create Readable Streamlit Inputs

In [28]:
# ===================================================================
# TRANSLATING TO HUMAN READABLE INPUTS (X_TRAIN CATEGORICAL COLUMNS)
# ===================================================================

# 1. Define the categorical columns and their valid human-readable values
categorical_categories = {
    'city_zone': [
        'Financial District', 'Industrial East', 'Suburban North', 
        'Downtown Core', 'Tech Park', 'Residential West'
    ],
    'road_type': [
        'Local Street', 'Arterial', 'Collector', 'Highway'
    ],
    'peak_period': [
        'Night', 'Off-Peak', 'Morning Peak', 'Midday', 'Evening Peak'
    ],
    'weather_condition': [
        'Clear', 'Rain', 'Heavy Rain', 'Fog'
    ],
    'road_condition': [
        'Poor', 'Good', 'Wet'
    ],
    'iot_sensor_health': [
        'Optimal', 'Degraded'
    ]
}

# 2. Build structured mapping dictionaries for each column
categorical_mappings = {}
for col, categories in categorical_categories.items():
    categorical_mappings[col] = {
        'options': categories,                               # Perfect for st.selectbox()
        'label_to_id': {label: idx for idx, label in enumerate(categories)}, # For encoding user input
        'id_to_label': {idx: label for idx, label in enumerate(categories)}  # For decoding model predictions if needed
    }

# 3. Export and save to a joblib file
joblib.dump(categorical_mappings, 'categorical_mappings.joblib')

print("Categorical mappings successfully compiled and saved to 'categorical_mappings.joblib'!")

Categorical mappings successfully compiled and saved to 'categorical_mappings.joblib'!


In [29]:
# ===================================================================
# TRANSLATING TO HUMAN READABLE INPUTS (X_TRAIN NUMERICAL COLUMNS)
# ===================================================================

# 1. Define feature classifications based on your schema
binary_features = [
    'is_weekend', 'is_holiday', 'rush_hour', 'nearby_school', 
    'nearby_hospital', 'nearby_market', 'construction_activity', 
    'public_event', 'accident_reported', 'emergency_vehicle_detected'
]

# Extract continuous features dynamically from your summary stats bounds (min, max, 50% median)
continuous_features_metadata = {
    'latitude': {'min': 40.636691, 'max': 40.790642, 'default': 40.721824},
    'longitude': {'min': -74.083673, 'max': -73.928036, 'default': -74.001825},
    'lanes': {'min': 2.0, 'max': 4.0, 'default': 3.0, 'step': 1.0},
    'speed_limit': {'min': 40.0, 'max': 90.0, 'default': 60.0, 'step': 5.0},
    'vehicle_count': {'min': 54.0, 'max': 3162.0, 'default': 308.0},
    'average_speed': {'min': 5.0, 'max': 90.0, 'default': 40.0},
    'heavy_vehicle_count': {'min': 1.0, 'max': 246.0, 'default': 16.0},
    'motorcycle_count': {'min': 3.0, 'max': 468.0, 'default': 32.0},
    'public_transport_count': {'min': 0.0, 'max': 156.0, 'default': 9.0},
    'traffic_density': {'min': 9.0, 'max': 378.5, 'default': 35.7},
    'queue_length': {'min': 0.0, 'max': 10111.6, 'default': 19.1},
    'average_wait_time': {'min': 0.0, 'max': 1174.5, 'default': 1.9},
    'hour': {'min': 0.0, 'max': 23.0, 'default': 11.5, 'step': 1.0},
    'day_of_week': {'min': 0.0, 'max': 6.0, 'default': 3.0, 'step': 1.0},
    'temperature': {'min': -2.8, 'max': 33.1, 'default': 15.0},
    'rainfall_mm': {'min': 0.0, 'max': 31.6, 'default': 0.0},
    'visibility_km': {'min': 0.5, 'max': 10.0, 'default': 9.9},
    'air_quality_index': {'min': 20.0, 'max': 152.0, 'default': 72.0},
    'parking_occupancy': {'min': 5.0, 'max': 100.0, 'default': 40.1},
    'traffic_camera_count': {'min': 1.0, 'max': 4.0, 'default': 2.0, 'step': 1.0},
    'signal_cycle_seconds': {'min': 60.0, 'max': 120.0, 'default': 60.0},
    'green_light_duration': {'min': 18.0, 'max': 84.0, 'default': 23.0},
    'traffic_flow_rate': {'min': 54.0, 'max': 3162.0, 'default': 308.0},
    'log_queue_length': {'min': 0.0, 'max': 10.0, 'default': 2.5} # Estimated log bounds based on queue length
}

# 2. Package numerical metadata
numerical_metadata = {
    'binary_features': binary_features,
    'continuous_features': continuous_features_metadata
}

# 3. Save to joblib
joblib.dump(numerical_metadata, 'numerical_metadata.joblib')

print("Numerical metadata successfully compiled and saved to 'numerical_metadata.joblib'!")

Numerical metadata successfully compiled and saved to 'numerical_metadata.joblib'!


## Create requirements.txt

In [ ]:
# List the exact package names your project relies on
project_packages = ['streamlit', 'joblib', 'numpy', 'pandas', 'xgboost', 'scikit-learn', 'plotly']

requirements = []
for pkg in project_packages:
    try:
        version = importlib.metadata.version(pkg)
        requirements.append(f"{pkg}=={version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"Warning: {pkg} is not installed in this environment.")

with open("requirements.txt", "w") as f:
    f.write("\n".join(requirements) + "\n")

print("requirements.txt generated successfully with exact local versions!\n")
for req in requirements:
    print(req)

requirements.txt generated successfully with exact local versions!

streamlit==1.60.0
joblib==1.5.3
numpy==2.4.4
pandas==3.0.3
xgboost==3.2.0
scikit-learn==1.9.0
plotly==6.7.0
